In [7]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("category_encoders") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "category-encoders"]
    )

# `course_helpers.py` sits beside the notebook, and Python does not search
# that folder on its own.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Standard library: `Counter` tallies how often a column is selected,
# `inspect` reads a selector's signature, `textwrap` folds printed text.
import inspect
import textwrap
import warnings
from collections import Counter

# Arrays, frames and plots.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

# Encoders for the categorical columns, and the two association statistics the
# typed filter scores a source column with.
from category_encoders import CountEncoder, TargetEncoder
from scipy.stats import chi2_contingency, pointbiserialr

# Feature extraction first, then the selectors this week compares.
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_selection import (
    RFE,
    RFECV,
    SelectFromModel,
    SelectKBest,
    SequentialFeatureSelector,
    VarianceThreshold,
)

# The imputer that fills each split, the model every technique is scored
# through, the splits it is scored on, and the score reported for it.
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit

# The scalers and encoders refitted inside every training split, and the
# warning the solver raises when it stops at `max_iter`.
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
OrdinalEncoder,
    RobustScaler,
    StandardScaler,
)
from sklearn.exceptions import ConvergenceWarning

# The course's shared cleaning log and plot colours.
from course_helpers import CleaningLog, PLOT_BLUE, PLOT_ORANGE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)  # For reproducibility
FIGSIZE = (6, 5)

# Convergence, one-hot and pandas deprecation warnings are silenced once here.
# The source of each is known and expected.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

ModuleNotFoundError: No module named 'course_helpers'

### Feature Engineering 

### Combination Features 

In [9]:
# A CSV stores no dtypes, so the six booleans, which carry gaps, would come
# back as `object`. The log's cast names the dtype each one should have.
cast = recipe_log.plan("6 boolean columns")["cast"]
 = pd.read_csv("../../data/interim/champions.csv", dtype=cast)
 = champions.set_index("RecordID")

X = champions.drop(columns=["Outcome", "Athlete Id"])
y = champions["Outcome"]

# Week 3's repeated 80/20 design
splitter = StratifiedShuffleSplit(n_splits=20, test_size=0.20,
                                  random_state=RANDOM_STATE)
splits = list(splitter.split(X, y))



NameError: name 'champions' is not defined

In [ ]:
def paired_delta(scores, reference):
    """Mean split-by-split difference from `reference`, its SEM, wins and verdict."""
    delta = np.asarray(scores) - np.asarray(reference)
    sem = delta.std(ddof=1) / np.sqrt(len(delta))
    # Two standard errors separate a difference from noise; a delta that is
    # zero on every split is no difference at all.
    verdict = ("identical on every split" if not delta.any()
               else "inside the noise" if abs(delta.mean()) <= 2 * sem
               else "better" if delta.mean() > 0 else "worse")
    return {"dF1": delta.mean(), "SEM": sem,
            "wins": f"{int((delta > 0).sum())}/{len(delta)}", "verdict": verdict}

In [ ]:
# 1. the sum of the eleven training-volume columns, beside the columns it sums
X_added = X.assign(total_training=X[log1p_columns].sum(axis=1))

# 2. two recipes: the sum added to the numeric columns, and the sum standing in
#    for the eleven
added = {**recipe, "numeric": numeric + ["total_training"],
         "log1p": log1p_columns + ["total_training"]}
replaced = {**recipe,
            "numeric": [c for c in numeric if c not in log1p_columns] + ["total_training"],
            "log1p": ["total_training"]}

# 3. each arm paired against the recipe's own run, on the same twenty splits
arms = pd.DataFrame({
    "added": paired_delta(heldout_f1(added, X_added)["F1"], reference["F1"]),
    "REPLACING the eleven": paired_delta(heldout_f1(replaced, X_added)["F1"],
                                         reference["F1"]),
}).T
print(arms.to_string(formatters={"dF1": "{:+.4f}".format, "SEM": "{:.4f}".format}))
     

## Feature Selection

### Filter Methods 

### Wrapper Methods 

### Embedded Methods 

### Feature Extraction

## Dimensionality Reduction 